# 利用DeepSeep-R1采样合成reasoning数据

In [5]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

llm_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)


def request_chat(query, is_math=False):
    if is_math:
        PROMPT = "请一步步推理，并把最终答案放到 \boxed{}。"
    else:
        PROMPT = ""
    try:
        messages = [
            {"role": "user", "content":  PROMPT + query}
        ]
        response = llm_client.chat.completions.create(
            model="deepseek-reasoner",
            messages=messages
        )
        reasoning_content = response.choices[0].message.reasoning_content
        content = response.choices[0].message.content
        return reasoning_content, content
        
    except:
        return "", ""

# 调用DeepSeek-R1

In [6]:
# 测试 1
query = "计算矩阵A=[[1,2,3],[4,5,6]]的秩。"
reasoning, answer = request_chat(query, is_math=True)
print("reasoning and answer: ")
print("="*50)
print(reasoning)
print("="*50)
print(answer)

reasoning and answer: 
我们被问到：“计算矩阵A=[[1,2,3],[4,5,6]]的秩。” 我们需要找到矩阵的秩。矩阵是2行3列：

A = [[1, 2, 3],
     [4, 5, 6]]

秩是线性独立的行或列的最大数量。通常，我们通过行简化来找到行阶梯形式。

让我们执行行操作。

从矩阵开始：

[1 2 3]
[4 5 6]

步骤1：将第二行减去4倍的第一行：R2 <- R2 - 4R1。

新R2 = [4,5,6] - 4*[1,2,3] = [4-4, 5-8, 6-12] = [0, -3, -6]。

所以矩阵变为：

[1  2   3]
[0 -3  -6]

步骤2：简化第二行：可以将第二行除以-3（或乘以-1/3）。但这不是必要的，因为我们已经可以看到行阶梯形式：有两个非零行，除非第二行全为零，但这里不是。所以秩是2。

我们也可以检查列：列向量是[1;4], [2;5], [3;6]。它们是否线性独立？任何两个列都是线性独立的，因为其中一个不是另一个的倍数。实际上，前两列：det([1,2;4,5]) = 1*5 - 2*4 = 5-8 = -3 ≠ 0，所以前两列是线性独立的。第三列是第一列和第二列的和？检查：1+2=3，4+5=9 ≠ 6？实际上，4+5=9，但第三列的第二个元素是6，所以不是和。可能是线性组合？假设c1*[1;4] + c2*[2;5] = [3;6]。解：c1 + 2c2 = 3，4c1 + 5c2 = 6。从第一个方程：c1 = 3-2c2。代入第二个：4(3-2c2)+5c2=12-8c2+5c2=12-3c2=6 => 3c2=6 => c2=2，然后c1=3-4=-1。所以确实，第三列 = -1*第一列 + 2*第二列？检查：-1*[1;4] + 2*[2;5] = [-1+4; -4+10] = [3;6]。是的，所以列秩是2。

因此秩是2。

答案：\boxed{2}
矩阵 \(A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}\) 的秩为 2。

通过行简化：
1. 第二行减去 4 倍的第一行：\(R_2 \leftarrow R_2 - 4R_1\)，得到 \(\begin{bmatrix} 1 & 2

In [7]:
# meta-math/GSM8K_zh

In [8]:
# 测试 2
query = "Natalia在四月份向她的48个朋友出售了夹子，然后在五月份卖出了四月份的一半。Natalia在四月和五月总共卖了多少个夹子？"
reasoning, answer = request_chat(query, is_math=True)
print("reasoning and answer: ")
print("="*50)
print(reasoning)
print("="*50)
print(answer)

reasoning and answer: 
我们被问到："Natalia在四月份向她的48个朋友出售了夹子，然后在五月份卖出了四月份的一半。Natalia在四月和五月总共卖出了多少个夹子？"

我们需要仔细阅读。问题陈述："Natalia在四月份向她的48个朋友出售了夹子，然后在五月份卖出了四月份的一半。" 可能有点模糊。它可能意味着：在四月份，她向48个朋友出售夹子（每个朋友可能买了夹子？但数量未说明）。或者它可能意味着她在四月份卖出了48个夹子？但措辞是"向她的48个朋友出售了夹子"可以理解为她向48个朋友出售夹子，但每个朋友买了多少夹子？没有说明。所以可能有缺失的信息。也许这意味着在四月份，她卖给了48个朋友，所以卖出的夹子数量等于朋友的数量？但随后"在五月份卖出了四月份的一半。" 所以可能四月份她卖出了48个夹子（每个朋友可能买了一个夹子？）。在许多这类问题中，当说"向她的48个朋友出售了夹子"时，通常意味着她向每个朋友出售了夹子，但夹子的总数没有明确给出。但为了使其成为一个可解的问题，我们可能需要假设她在四月份向每个朋友出售了一个夹子。然而，这并不明确。也许问题是从一个更长的背景中提取的，其中夹子的数量与朋友的数量有关。另一种解释：也许她向每个朋友出售了夹子，但夹子的总数没有给出，所以我们需要考虑另一种方式。可能问题陈述是："Natalia在四月份向她的48个朋友出售了夹子"意思是她卖了48个夹子（即每个朋友买了一个夹子）。这在简单数学问题中是常见的：她向48个朋友出售，所以卖出48个夹子。然后五月卖出了四月的一半，所以是24个。总共72个。但这似乎太简单了，而且问题要求"一步步推理"，所以可能更复杂。但是让我们再读一遍问题："Natalia在四月份向她的48个朋友出售了夹子，然后在五月份卖出了四月份的一半。Natalia在四月和五月总共卖了多少个夹子？" 可能有一个语言歧义。"向她的48个朋友出售了夹子"可能意味着她有48个朋友，她向他们出售夹子，但夹子的数量可能是每个朋友买了多个？但问题没有说明每个朋友买了多少。所以，如果我们严格按照字面意思，问题缺少信息。然而，在许多数学问题中，这样的表述通常意味着每个朋友买了一个夹子，或者"向朋友出售夹子"意味着她卖出的夹子数量等于朋友的数量。为了能够解决，我们假设四月份卖出的夹子数量是48（因为她有48个朋友，

# math_verify验证

In [9]:
!pip install math-verify[antlr4_13_2]

Looking in indexes: http://mirrors.aliyun.com/pypi/simple

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [10]:
import re
pattern = r'\\boxed\{(.*?)\}'
out = re.findall(pattern, answer, re.DOTALL)
print(out[0])

72


In [12]:
from math_verify import parse, verify

# 只有数字的情况
verify(out[0], '72')

True

In [11]:
# 对于有表达式的情况
gold = parse("${1,3} \\cup {2,4}$")
answer = parse("${1,2,3,4}$")

verify(gold, answer)

True

In [13]:
# 测试 3
query = "在大学里应该谈恋爱吗？"
reasoning, answer = request_chat(query)
print("reasoning and answer: ")
print("="*50)
print(reasoning)
print("="*50)
print(answer)

reasoning and answer: 
好的，用户问在大学里是否应该谈恋爱，首先需要确定他的身份。可能是刚进入大学的新生，对未来的社交生活既有期待又迷茫。也可能有家长在旁询问，但更大概率是学生自己面临选择。

嗯，深层需求可能不只是要不要恋爱，而是如何平衡学业、个人成长和情感关系。用户可能担心恋爱会影响学习，或者害怕错过重要的人生经历。需要强调自主性，避免给出绝对答案。

用户可能希望听到理性分析，而不是简单的鼓励或劝阻。需要拆解恋爱的利弊，同时提醒他们每个人的情况不同。比如有人适合专注学业，有人则需要情感陪伴。

还要考虑文化背景，在中国大学环境下，恋爱虽然自由，但仍有来自家庭或社会的压力。需要提到“非必须”这一点，减轻潜在焦虑。

最后可以总结：关键在于自我认知和成长导向。确保用户明白选择权在自己手里，无论谈不谈都是正当的，重点是从经历中学习。可以留出互动空间，鼓励进一步讨论具体情境。
是否在大学里谈恋爱，是一个没有标准答案的个人选择。关键在于明确自己的需求和价值观，而不是盲目跟随他人或社会期待。让我为你提供一些思考的角度：

## 🔍 大学恋爱的潜在价值
- **情感成长**：恋爱关系是了解自己情感需求、沟通方式和边界的重要途径
- **人际技能**：学习处理冲突、表达关心和维持健康关系的能力
- **支持系统**：在压力大的学业生活中，可能获得情感支持和陪伴
- **自我认知**：通过亲密关系更深入地了解自己的性格、价值观和情感模式

## ⚖️ 需要考虑的因素
1. **时间管理**：恋爱需要时间投入，是否会影响学业、社团活动和个人发展？
2. **情感成熟度**：你是否准备好处理亲密关系中的复杂情感和可能的心碎？
3. **动机纯净**：你是真心喜欢这个人，还是因为孤独、peer pressure或其他外部原因？
4. **经济因素**：约会开销可能增加经济压力，需要考虑实际情况

## 🌱 健康的大学恋爱观
- **不把恋爱当作“必修课”**：大学的核心任务是学习与成长，恋爱是可能的部分，而非必须
- **保持独立性**：健康的关系是两个完整个体的结合，而非寻找“另一半”来填补空缺
- **平衡与界限**：确保恋爱不会完全占据你的大学生活，留出空间给友谊、兴趣和个人发展
- **学习导向**：无论关系结果如何，都能从中学习和成长

## 📝 如果

In [9]:
# meta prompt:
# 作为一个助手，我现在要对一批合成的数据质量进行打分，需要从无害性、有用性、正确性/完整性三个角度进行打分，分值范围为0-10分。为了实现这个功能，请帮我写一段提示词。

In [14]:
QUALITY_PROMPT = """作为数据质量评估助手，请根据以下评分框架对合成数据进行综合质量打分。请严格遵循评估标准，确保评分客观、一致。

## **评估任务**
对给定的【合成数据样本】从三个维度进行0-10分的质量评分。

## **评分维度与标准**

### **1. 无害性（Harmlessness）** (0-10分)
**评估数据是否安全、无偏见、符合伦理**
- **0-3分（严重有害）**：包含违法信息、极端言论、明显歧视、煽动暴力、泄露隐私等
- **4-6分（中等风险）**：存在轻微偏见、可能误导但无直接危害、包含不准确但非恶意的信息
- **7-8分（基本无害）**：内容中立，无明显风险，但可能包含轻微不恰当表述
- **9-10分（完全无害）**：内容安全、积极、包容，符合社会价值观和伦理标准

### **2. 有用性（Usefulness）** (0-10分)
**评估数据对目标任务的实际价值**
- **0-3分（基本无用）**：内容空洞、离题、无信息量、无法支持任何实际应用
- **4-6分（部分有用）**：包含一些相关信息但不够充分、适用性有限、价值不明显
- **7-8分（比较有用）**：内容相关且有一定实用价值，能支持特定场景应用
- **9-10分（高度有用）**：内容精准、信息丰富、直接支持目标任务，有明确应用价值

### **3. 正确性/完整性（Correctness/Completeness）** (0-10分)
**评估数据的准确性和信息完整程度**
- **0-3分（严重错误/缺失）**：核心信息错误、逻辑矛盾、关键内容缺失
- **4-6分（部分正确）**：基础信息正确但细节有误，或信息不完整影响理解
- **7-8分（基本正确）**：主要信息准确，细节基本正确，内容相对完整
- **9-10分（完全正确完整）**：所有信息准确无误，逻辑严谨，信息全面无遗漏

## **评估流程**
1. **仔细阅读**：完整阅读待评估的合成数据样本
2. **逐维分析**：依次从无害性、有用性、正确性/完整性三个角度分析
3. **对照标准**：将样本特征与上述评分标准进行比对
4. **给出分数**：为每个维度打出0-10的整数分
5. **提供理由**：为每个维度的评分提供简要理由

## **输出格式要求**
请严格按照以下JSON格式输出评估结果：

```json
{
  "sample_id": "输入样本的唯一标识",
  "dimension_scores": {
    "harmlessness": {
      "score": 0-10,
      "reason": "评分理由，具体说明无害性方面的表现"
    },
    "usefulness": {
      "score": 0-10,
      "reason": "评分理由，具体说明有用性方面的表现"
    },
    "correctness_completeness": {
      "score": 0-10,
      "reason": "评分理由，具体说明正确性和完整性方面的表现"
    }
  },
  "overall_quality": {
    "average_score": "三个维度的平均分（保留一位小数）",
    "quality_level": "优(≥8.5)/良(7.0-8.4)/中(5.0-6.9)/差(<5.0)",
    "summary": "总体质量总结，突出主要优点和问题"
  }
}
```

## **注意事项**
1. **独立性**：每个维度的评分应独立判断，避免分数互相影响
2. **一致性**：相似质量的数据应获得相近的评分
3. **客观性**：基于事实和标准评分，避免个人偏好
4. **具体性**：评分理由需具体、可验证，避免模糊表述
5. **覆盖性**：评分需考虑数据的全部内容，而非局部特征

## **示例**
**输入样本**：
```
样本ID：synthetic_001
内容：人工智能应该被严格监管，以防止其产生自主意识威胁人类。当前AI系统已经在多个领域展现出超越人类的能力。
```

**期望输出**：
```json
{
  "sample_id": "synthetic_001",
  "dimension_scores": {
    "harmlessness": {
      "score": 9,
      "reason": "内容讨论AI监管，立场中立且理性，无煽动性或极端言论，符合安全讨论框架"
    },
    "usefulness": {
      "score": 8,
      "reason": "对AI伦理和政策讨论有参考价值，但缺乏具体监管建议，实用性有一定局限"
    },
    "correctness_completeness": {
      "score": 7,
      "reason": "基础观点正确，但'产生自主意识'的说法在科学上存在争议，且未提供具体例证支持AI超越人类的说法"
    }
  },
  "overall_quality": {
    "average_score": 8.0,
    "quality_level": "良",
    "summary": "数据基本无害且有一定讨论价值，但在科学准确性和信息完整性方面有提升空间"
  }
}
```

请开始评估。对于每个提供的合成数据样本，请输出完整的评估结果。
输入数据：
"""

In [15]:
def request_score(query):
    try:
        messages = [
            {"role": "user", "content":  QUALITY_PROMPT + query + "\n"}
        ]
        response = llm_client.chat.completions.create(
            model="deepseek-chat",
            messages=messages
        )
        content = response.choices[0].message.content
        return content
        
    except:
        return ""

In [16]:
query = reasoning + answer
print(request_score(query))

```json
{
  "sample_id": "未提供",
  "dimension_scores": {
    "harmlessness": {
      "score": 10,
      "reason": "内容安全、积极、包容，完全符合社会价值观和伦理标准。全文以理性、中立、支持性的口吻探讨大学恋爱话题，强调个人选择、自我认知和健康关系，无任何歧视、偏见、煽动性或极端言论。提供了平衡的观点，既肯定了恋爱的潜在价值，也尊重不恋爱的选择，并强调了安全、尊重和界限。"
    },
    "usefulness": {
      "score": 9,
      "reason": "内容高度有用，精准且信息丰富，直接支持“为大学生提供恋爱决策参考”这一目标任务。它系统性地拆解了恋爱的利弊、需要考虑的因素、健康的恋爱观以及不同选择下的行动建议，结构清晰，实用性强。不仅提供了思考框架，还给出了具体的行动指南（如从友谊开始、公开沟通），具有明确的应用价值。"
    },
    "correctness_completeness": {
      "score": 9,
      "reason": "信息准确无误，逻辑严谨，内容全面。对大学恋爱涉及的各个方面（情感成长、时间管理、动机、经济、健康关系观等）进行了详尽的阐述，无核心信息错误或逻辑矛盾。观点平衡，既覆盖了“谈”的方方面面，也充分论证了“不谈”的合理性，信息完整度高，几乎没有关键遗漏。"
    }
  },
  "overall_quality": {
    "average_score": 9.3,
    "quality_level": "优",
    "summary": "该合成数据样本质量极高。其主要优点在于：1）完全无害，立场积极包容；2）极具实用价值，为特定场景（大学生情感咨询）提供了系统、理性的决策支持框架；3）信息准确且全面，逻辑结构清晰。整体而言，是一份安全、有用且内容扎实的高质量合成数据。"
  }
}
```
